## True Parallelism (Bypassing the GIL)

**First Principle:** Python's C-level Global Interpreter Lock (GIL) restricts execution to one thread at a time to prevent memory corruption. To achieve true CPU parallelism and bypass the GIL, you must completely separate the memory spaces. Spawning new operating system processes achieves this because each new process gets its own Python interpreter, its own isolated private memory heap, and its own independent GIL.

> **Analogy:** If threads are multiple chefs fighting over a single ledger pen (the GIL) in one kitchen, multiprocessing is building entirely separate physical kitchens next door. Each new kitchen has its own staff, its own pantry (memory), and its own ledger, eliminating resource contention.

```text
                             +------------------------+
                             |    PRIMARY PROCESS     |
                             |   (Dispatches Tasks)   |
                             +------------------------+
                                         |
                +------------------------+------------------------+
                |                                                 |
                v                                                 v
  +---------------------------+                     +---------------------------+
  |     WORKER PROCESS 1      |                     |     WORKER PROCESS 2      |
  |  - Own Virtual Memory     |                     |  - Own Virtual Memory     |
  |  - Own CPython Runtime    |                     |  - Own CPython Runtime    |
  |  - Own Independent GIL    |                     |  - Own Independent GIL    |
  +---------------------------+                     +---------------------------+
                |                                                 |
                v                                                 v
       [ CPU Physical Core 0 ]                           [ CPU Physical Core 1 ]

- **Inter-Process Communication (IPC):** Because memory heaps are strictly isolated between processes, data shared between processes must be serialized (pickled), piped across system sockets, and deserialized in the worker process.

- **The Trade-off:** Multiprocessing provides true parallel CPU execution, but it comes with a much higher memory overhead since the entire Python environment is duplicated for each process.

In [ ]:
import multiprocessing
import time


def cpu_heavy_math(name):
    """A pure CPU-bound task that would normally be blocked by the GIL."""

    # This bypasses the GIL because it runs in a completely separate OS process
    total = sum(range(50))

    return f"Process {name} completed with sum: {total}"


# The __main__ guard is mandatory in multiprocessing to prevent recursive spawning
if __name__ == "__main__":
    print("=== MULTIPROCESSING: TRUE PARALLELISM ===")

    start_time = time.time()

    # Spawn a pool of worker processes to distribute the tasks
    with multiprocessing.Pool(processes=2) as pool:

        # pool.map automatically handles the IPC and data serialization
        results = pool.map(cpu_heavy_math, ["A", "B"])

    for res in results:
        print(f" -> {res}")

    print(f"Parallel Execution Time: {time.time() - start_time:.2f} seconds")

=== MULTIPROCESSING: TRUE PARALLELISM ===
